# WebBaseLoader
> 웹페이지(URL)의 텍스트를 가져와 Documnet로 변환 

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_path="https://www.example.com/", # 수집할 웹 페이지 url

    header_template={
        # 서버가 차단하지 않도록 브라우저처럼 보이게 user-agent 헤더 설정
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    },
    verify_ssl=False, # SSL 인증서 검증 비활성화 = 테스트용 
    requests_kwargs={"timeout":10} # 요청 타임아웃 10초 설정
)

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
docs = loader.load()

print(f"로드된 문서의 수:{len(docs)}")

로드된 문서의 수:1


In [4]:
print(f"첫번째 문서의 메타정보 확인:\n{docs[0].metadata}")

첫번째 문서의 메타정보 확인:
{'source': 'https://www.example.com/', 'title': 'Example Domain', 'language': 'en'}


In [5]:
print(f"첫번째 문서의 데이터 확인:\n{docs[0].page_content}")

첫번째 문서의 데이터 확인:
Example DomainExample DomainThis domain is for use in documentation examples without needing permission. Avoid use in operations.Learn more



# Multiple pages 

In [11]:
from langchain_community.document_loaders import WebBaseLoader

# webBaseLoader를 이용하여 여러 웹 페이지를 한 번에 로드하는 로더 생성
loader = WebBaseLoader(
    web_paths=["https://www.example.com/", "https://google.com"] # 수집할 여러 웹 페이지 url 리스트

)
# 지정된 url 들에서 html을 가져와 LangChain Document 형태로 로드 
docs = loader.load()
# 로드된 Document 객체의 개수 출력 
print(f"로드된 문서의 수 : {len(docs)}")

SSLError: HTTPSConnectionPool(host='www.example.com', port=443): Max retries exceeded with url: / (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1032)')))

In [12]:
for i, doc in enumerate(docs):
    print("="*50)
    print(f"Document {i+1}:")
    print(f"   - 타입: {type(doc)}")
    print(f"   - page_content 타입: {type(doc.page_content)}")
    print(f"   - metadata 타입: {type(doc.metadata)}")
    print(f"   - metadata 내용: {doc.metadata}")
    print(f"   - 내용 길이: {len(doc.page_content)} 문자")
    print()

Document 1:
   - 타입: <class 'langchain_core.documents.base.Document'>
   - page_content 타입: <class 'str'>
   - metadata 타입: <class 'dict'>
   - metadata 내용: {'source': 'https://www.example.com/', 'title': 'Example Domain', 'language': 'en'}
   - 내용 길이: 140 문자



# Loader with bs4 

In [ ]:
import requests
from bs4 import BeautifulSoup as bs

def main():
    # 서버 차단을 피하기 위해 실제 브라우저처럼 보이게 설정한 커스텀 헤더
    custom_header = {
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
        "referer": "https://pann.ante.com/"   # 요청이 어디서 온 것처럼 보이게 하는 Referer 값
    }

    # 크롤링 대상 URL
    url = "https://pann.nate.com/talk/350939697"

    # 해당 URL로 GET 요청을 보내 HTML을 가져옴
    response = requests.get(url, headers=custom_header)

    # 요청이 실패한 경우 예외 발생 (4xx, 5xx 에러를 잡기 위해 사용)
    response.raise_for_status()

    # BeautifulSoup을 이용해 HTML 문서를 파싱
    soup = bs(response.text, "html.parser")

    # 파싱된 soup 객체 반환
    return soup


In [14]:
soup = main()

# 네이트 판 댓글 추출 

#### BS4를 사용한 특정 요소 추출 예제

In [15]:
def crawlig(soup):
    # HTML 문서에서 usertxt 요소들을 모두 찾음 = 댓글이 담겨있는 태그 
    dd_list = soup.find_all("dd",class_="usertxt")

    # 찾은 모든 댓글 태그를 순회하며 인덱스와 함께 처리 
    for idx, dd in enumerate(dd_list):
        # 태그 내부의 텍스트만 추출하고, 불필요한 줄바꿈/ 탭 제거
        comment = dd.get_text().replace("\n","").replace("\t","")

        # 댓글 번호와 내용을 출력 
        print(f"{idx}번째 댓글\n >> {comment}")

In [17]:
crawlig(soup)

0번째 댓글
 >>  우리 댕댕이ㅋ 
1번째 댓글
 >> 우리집도 말티 댕댕이들은 사랑입니당! 
2번째 댓글
 >> ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ 다이쁨 완전 얼짱이넹
3번째 댓글
 >> 우리 딸랑이 
4번째 댓글
 >> 말티는 사랑입니당^^ 
5번째 댓글
 >> 너무 귀엽당^^
6번째 댓글
 >> 밑에서 두번째 사진 털때문인지 뾰루퉁한거 넘 귀여워요 ㅋㅋㅋㅋㅋ!울집강아지도 털때문에 가끔 눈이 화난눈 되거든요 ㅎㅎㅎㅎ 귀엽 ㅠ.ㅠ*
7번째 댓글
 >> 우리집도 
8번째 댓글
 >> 달릴 때 졸귘ㅋㅋ 눈이랑 코랑 동글동글 너무 귀엽 ㅠㅠㅠㅠㅠ
9번째 댓글
 >> 안녕 칭구 
10번째 댓글
 >> 개똥냄새 쩔게 생겼네
11번째 댓글
 >> 안녕 난 두부야 
12번째 댓글
 >> 아 너모 이쁘다. 항상 건강하자 아가
13번째 댓글
 >> 힐링하고 갑니다❤️
14번째 댓글
 >> 안뇽 ? 
15번째 댓글
 >> 네츄럴 부스스.. 예쁘네요..^^
16번째 댓글
 >> 으악 싑알 넘귀여워 말티즈 사랑해말티즈최고 말티즈 너무귀엽다
17번째 댓글
 >> 순백의 청순함에서 깨발랄 ㅋ귀엽네요
18번째 댓글
 >> 우리 댕댕이ㅋ 
19번째 댓글
 >> 어머나!


#### WebBaseLoader로 네이트 판 댓글 추출 

In [18]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

# 네이트 판 댓글만 추출하기 위한 WebBaseLoader 설정
loader = WebBaseLoader(
    # 가져올 웹 페이지 주소 (네이트 판 특정 게시글)
    web_path="https://pann.nate.com/talk/350939697",
    
    # HTTP 요청 시 사용할 헤더 설정 (브라우저처럼 보이도록)
    header_template={
        "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
        "referer": "https://pann.nate.com/"  # 요청의 출처를 표시하여 차단 우회
    },
    
    # BeautifulSoup 파서 설정
    bs_kwargs={
        # SoupStrainer: 특정 태그만 골라서 빠르게 파싱하도록 제한
        "parse_only": SoupStrainer(
            'dd',                          # dd 태그만 파싱
            attrs={'class': 'usertxt'}     # class="usertxt" 인 dd만 추출 (댓글 본문)
        )
    },
    
    # BeautifulSoup get_text() 옵션 설정
    bs_get_text_kwargs={
        "separator": "\n",  # 텍스트 사이 구분자: 여러 요소가 있으면 줄바꿈으로 구분
        "strip": True       # 앞뒤 공백 제거
    },
    
    # HTTP 요청 옵션 설정
    requests_kwargs={
        "timeout": 10,      # 요청 타임아웃 10초
        "verify": False     # SSL 인증서 검증 비활성화 (테스트 환경에서만 사용 권장)
    }
)


In [19]:
# 설정한 로드를 이용해 웹 페이지에서 문서 (댓글들) 추출 
docs = loader.load()

# 로드된 Document 객체 수 출력 
print(f"로드된 문서의 수:{len(docs)}")

로드된 문서의 수:1


In [20]:
# 추출된 댓글 내용 확인 
print("추출된 댓글 내용!")
print("="*50)
print(docs[0].page_content)

추출된 댓글 내용!
우리 댕댕이ㅋ
우리집도 말티 
댕댕이들은 사랑입니당!
ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ 다이쁨 완전 얼짱이넹
우리 딸랑이
말티는 사랑입니당^^
너무 귀엽당^^
밑에서 두번째 사진 털때문인지 뾰루퉁한거 넘 귀여워요 ㅋㅋㅋㅋㅋ!
울집강아지도 털때문에 가끔 눈이 화난눈 되거든요 ㅎㅎㅎㅎ 귀엽 ㅠ.ㅠ*
우리집도
달릴 때 졸귘ㅋㅋ 눈이랑 코랑 동글동글 너무 귀엽 ㅠㅠㅠㅠㅠ
안녕 칭구
개똥냄새 쩔게 생겼네
안녕 난 두부야
아 너모 이쁘다. 항상 건강하자 아가
힐링하고 갑니다❤️
안뇽 ?
네츄럴 부스스.. 예쁘네요..^^
으악 싑알 넘귀여워 말티즈 사랑해말티즈최고 말티즈 너무귀엽다
순백의 청순함에서 깨발랄 ㅋ
귀엽네요
우리 댕댕이ㅋ
어머나!
